# Embedding ResNet50 + cosine
        **vision › retrieval** · notebook `resnet50_cosine`

        **Kapan dipakai:** Pencarian gambar mirip dengan fitur ImageNet; sederhana dan kuat.

        **Jebakan umum:** Embedding klasifikasi, bukan dilatih untuk similarity; fine-tuning metric learning lebih baik.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** CIFAR-10 subset (4000 latih / 1000 uji, 10 kelas, 32x32). Galeri = gambar latih; query = gambar uji; **relevan** = kelas yang sama.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
DATA_ROOT = ROOT / "data_latihan/vision/classification/cifar10_small"; FEAT = "resnet50"; SIZE = 96
N_QUERY, TOPK, SEED = 300, 10, 42
MODEL_KEY = "resnet50_cosine"
OUT = out_dir("vision", "retrieval", MODEL_KEY); set_seed(SEED)

## 2. Data

In [ ]:
from superdb.vision import load_images, show_grid
tr = pd.read_csv(DATA_ROOT / "train.csv"); te = pd.read_csv(DATA_ROOT / "test.csv").merge(pd.read_csv(DATA_ROOT / "test_labels.csv"), on="id").sample(N_QUERY, random_state=SEED)
G = load_images(tr["path"], DATA_ROOT); Q = load_images(te["path"], DATA_ROOT); g_lab, q_lab = tr["label"].to_numpy(), te["label"].to_numpy()
print(G.shape, Q.shape)

## 3. Embedding galeri dan query

In [ ]:
from superdb.vision import extract_features
from sklearn.preprocessing import normalize
Eg, Eq = normalize(extract_features(FEAT, G, SIZE)), normalize(extract_features(FEAT, Q, SIZE))

## 4. Metrik ranking
**P@10**: dari 10 hasil teratas berapa yang sekelas. **MAP@10**, **nDCG@10**: memperhitungkan urutan. **MRR**: posisi hasil relevan pertama. Garis dasar acak = 10%.

In [ ]:
from superdb.text_extra import retrieval_metrics
scores = Eq @ Eg.T
metrics = retrieval_metrics(scores, q_lab, g_lab, k=TOPK); display(show_table({**metrics, "acak P@k (dasar)": float(np.mean([np.mean(g_lab == l) for l in q_lab]))}))

## 5. Contoh: query dan 6 hasil teratas (hijau = sekelas)

In [ ]:
fig, ax = plt.subplots(3, 7, figsize=(11, 5))
for r in range(3):
    ax[r, 0].imshow(Q[r]); ax[r, 0].set_title(f"QUERY\n{q_lab[r]}", fontsize=8)
    for c, j in enumerate(np.argsort(-scores[r])[:6], 1):
        ax[r, c].imshow(G[j]); ax[r, c].set_title(g_lab[j], fontsize=8, color="green" if g_lab[j] == q_lab[r] else "red")
    [a.axis("off") for a in ax[r]]
plt.tight_layout(); plt.savefig(OUT / "retrieval.png", dpi=130); plt.show()
save_json({"metrics": metrics, "feat": FEAT}, OUT / "metrics.json"); np.save(OUT / "gallery_embeddings.npy", Eg.astype("float32")); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan HOG, ResNet, ConvNeXt, ViT, CLIP pada query yang sama: embedding semantik jauh lebih baik dari fitur manual.
- Galeri besar: gunakan indeks aproksimasi (faiss/annoy) atau `sklearn.neighbors` dengan `algorithm='brute'` pada GPU.
- Fine-tuning dengan metric learning (triplet loss) dapat menaikkan skor lagi.